In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())
print("\nLabel counts:")
print(df.iloc[:, -1].value_counts())

Shape: (651191, 3)

Columns: ['url', 'type', 'label']

First 5 rows:
                                                 url        type  label
0                                   br-icloud.com.br    phishing      1
1                mp3raid.com/music/krizz_kaliko.html      benign      0
2                    bopsecrets.org/rexroth/cr/1.htm      benign      0
3  http://www.garage-pirenne.be/index.php?option=...  defacement      0
4  http://adventure-nicaragua.net/index.php?optio...  defacement      0

Label counts:
label
0    557080
1     94111
Name: count, dtype: int64


In [22]:
import sys 
print(sys.executable)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import re, math
from urllib.parse import urlparse, unquote

df = pd.read_csv(r"C:\Users\puthi\OneDrive - Swinburne University\Documents\malicious_phish.csv") 

df["label"] = (df["type"] == "phishing").astype(int)

print("Total rows:", len(df))
print("\nBinary label counts:")
print(df["label"].value_counts())
print("\nOriginal type counts:")
print(df["type"].value_counts())
print("\nMissing URLs:", df["url"].isna().sum())
print("Duplicate URLs:", df["url"].duplicated().sum())

c:\Users\puthi\AppData\Local\Programs\Python\Python313\python.exe
Total rows: 651191

Binary label counts:
label
0    557080
1     94111
Name: count, dtype: int64

Original type counts:
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

Missing URLs: 0
Duplicate URLs: 10072


In [23]:
#drop missing labels 
df = df.dropna(subset=["url"]).reset_index(drop=True)

#drop duplicate URLs
before = len(df)
df = df.drop_duplicates(subset="url").reset_index(drop=True)
after = len(df)
print(f"Removed {before - after} duplicate URLs")
print(f"Remaining rows: {after}")
print(df["label"].value_counts())


Removed 10072 duplicate URLs
Remaining rows: 641119
label
0    547033
1     94086
Name: count, dtype: int64


In [26]:
def normalize(url):
    #Normalizes the URL into a standard format for processing

    if not isinstance(url, str):
        return ""
    url = url.strip()
    try:
        url = unquote(url)
    except Exception: 
        pass
    if "://" not in url:
        url = "http://" + url
    try:
        parsed = urlparse(url)
        scheme = parsed.scheme.lower()
        host = parsed.netloc.lower()
        path = parsed.path 
        query = parsed.query 
        return f"{scheme}://{host}{path}" + (f"?{query}" if query else "")
    except ValueError:
        return ""

df["url_clean"] = df["url"].apply(normalize)
print("After normalization. Columns:", df.columns.tolist())
print(df[["url", "url_clean"]].head(10).to_string())


After normalization. Columns: ['url', 'type', 'label', 'url_clean']
                                                                                                                                                                                                                                           url                                                                                                                                                                                                                                    url_clean
0                                                                                                                                                                                                                             br-icloud.com.br                                                                                                                                                                                                                      http

In [27]:
df.to_csv("df_normalized.csv", index=False)
print("Saved df_normalized.csv")
print("Rows:", len(df), " Columns:", df.columns.tolist())
print("URLs that failed to parse:", (df["url_clean"] == "").sum())

Saved df_normalized.csv
Rows: 641119  Columns: ['url', 'type', 'label', 'url_clean']
URLs that failed to parse: 17


In [33]:
suspicious_words = [
    "verify", "account", "login", "update", "secure", "banking", "confirm", 
    "password", "security", "signin", "support", "service", "auth", "authenticate",
    "winner", "prize", "websrc"
]

suspicious_tlds = {
    "tk", "ml", "ga", "xyz", "top", "icu", "sbs", "cfd", "zip", "move"
}

companies = ["amazon", "paypal", "microsoft","apple", "twitter", "facebook", "google", "netflix", "bank"]


def extract_features(url):
    parsed = urlparse(url)
    host = parsed.netloc
    path = parsed.path
    url_lower = url.lower()
    host_no_port = host.split(":")[0]


    has_ip = int(bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host_no_port)))

    parts = host_no_port.split(".")
    tld = parts[-1] if len(parts) > 1 else ""

    companies_in_sub = int(any(c in ".".join(parts[:-2]) for c in companies) and len(parts) > 2)

    return {
        "url_length": len(url),
        "host_length": len(host),
        "path_length": len(path),
        "n_hyphens": url.count("-"),
        "n_dots": url.count("."),
        "n_digits": sum(c.isdigit() for c in url),
        "n_slashes": url.count("/"),
        "n_subdomains": max(0, host_no_port.count(".") - 1),
        "has_ip": has_ip,
        "has_https": int(parsed.scheme == "https"),
        "has_at" : int("@" in url),
        "has_port": int(":" in host),
        "suspicious_words": sum(w in url_lower for w in suspicious_words),
        "digit_ratio": sum(c.isdigit() for c in url) / max(len(url), 1),
        "suspicious_tlds": int(tld in suspicious_tlds),
        "companies_in_sub": companies_in_sub,
        "tld_length": len(tld),

    }

test_urls = [
    "https://www.google.com",
    "http://192.153.1.1/login/verify-account.php",

]

for u in test_urls:
    print("URL:", u)
    for k, v in extract_features(u).items():
        print(f" {k:20s} = {v}")
    print()


URL: https://www.google.com
 url_length           = 22
 host_length          = 14
 path_length          = 0
 n_hyphens            = 0
 n_dots               = 2
 n_digits             = 0
 n_slashes            = 2
 n_subdomains         = 1
 has_ip               = 0
 has_https            = 1
 has_at               = 0
 has_port             = 0
 suspicious_words     = 0
 digit_ratio          = 0.0
 suspicious_tlds      = 0
 companies_in_sub     = 0
 tld_length           = 3

URL: http://192.153.1.1/login/verify-account.php
 url_length           = 43
 host_length          = 11
 path_length          = 25
 n_hyphens            = 1
 n_dots               = 4
 n_digits             = 8
 n_slashes            = 4
 n_subdomains         = 2
 has_ip               = 1
 has_https            = 0
 has_at               = 0
 has_port             = 0
 suspicious_words     = 3
 digit_ratio          = 0.18604651162790697
 suspicious_tlds      = 0
 companies_in_sub     = 0
 tld_length           = 1



In [ ]:
#testing extract_ffeatures with a risky URL
test = extract_features("http://paypal.com.evil.ru/login")
print(test)

{'url_length': 31, 'host_length': 18, 'path_length': 6, 'n_hyphens': 0, 'n_dots': 3, 'n_digits': 0, 'n_slashes': 3, 'n_subdomains': 2, 'has_ip': 0, 'has_https': 0, 'has_at': 0, 'has_port': 0, 'suspicious_words': 1, 'digit_ratio': 0.0, 'suspicious_tlds': 0, 'companies_in_sub': 1, 'tld_length': 2}


In [ ]:
#This applies extract features to every URL in the dataset, turning it into a numeric feature table 

import time
start = time.time()

features_df = df["url_clean"].apply(extract_features).apply(pd.Series)

df_full = pd.concat([df.reset_index(drop=True), features_df], axis=1)

elapsed = time.time() - start
print(f"Done in {elapsed:.1f} seconds")
print("Final shape:", df_full.shape)
print("Columns:", df_full.columns.tolist())
print()
print("First 3 rows (feature columns only):")
print(df_full[features_df.columns.tolist()].head(3).to_string())

Done in 107.5 seconds
Final shape: (641119, 21)
Columns: ['url', 'type', 'label', 'url_clean', 'url_length', 'host_length', 'path_length', 'n_hyphens', 'n_dots', 'n_digits', 'n_slashes', 'n_subdomains', 'has_ip', 'has_https', 'has_at', 'has_port', 'suspicious_words', 'digit_ratio', 'suspicious_tlds', 'companies_in_sub', 'tld_length']

First 3 rows (feature columns only):
   url_length  host_length  path_length  n_hyphens  n_dots  n_digits  n_slashes  n_subdomains  has_ip  has_https  has_at  has_port  suspicious_words  digit_ratio  suspicious_tlds  companies_in_sub  tld_length
0        23.0         16.0          0.0        1.0     2.0       0.0        2.0           1.0     0.0        0.0     0.0       0.0               0.0     0.000000              0.0               0.0         2.0
1        42.0         11.0         24.0        0.0     2.0       1.0        4.0           0.0     0.0        0.0     0.0       0.0               0.0     0.023810              0.0               0.0         3.0

In [32]:
#save feature table to CSV
df_full.to_csv("features.csv", index=False)
print("Saved features.csv, Shape:)", df_full.shape)

Saved features.csv, Shape:) (641119, 21)
